In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


cargar schema para utilizarlo en el prompt

In [2]:
import json

file_path = '/content/drive/MyDrive/programacion_funcional/STLC_AST_Schema.json'

with open(file_path, 'r', encoding='utf-8') as f:
    schema = json.load(f)

# Verificar el contenido
print(type(schema))
print(list(schema.keys()) if isinstance(schema, dict) else schema[:2])

<class 'dict'>
['$schema', 'title', 'description', '$ref', 'definitions']


# prompt

**Documentación Técnica: Pipeline de Compilación NL a STLC AST**

Este módulo documenta en detalle el diseño del prompt maestro, el esquema conversacional few-shot y la implementación de las funciones utilizadas para compilar reglas de negocio expresadas en lenguaje natural a un Árbol de Sintaxis Abstracta (AST) basado en el Cálculo Lambda Simplemente Tipado (STLC), validado contra el esquema formal STLC_AST_Schema.json.


**1) Justificación y Arquitectura del Prompt**

*1.1 Variable system_instruction*

Define el contrato de compilación determinista y delimita el comportamiento del modelo mediante directivas imperativas:

Inyección del Esquema Formal: Se incluye la serialización en texto del esquema (schema_str) dentro de los delimitadores <JSON_SCHEMA>. Esto expone al modelo los tipos primitivos válidos (Double, Int, Bool, String) y las definiciones recursivas de Expr y Type.

Directiva de Exhaustividad Semántica (REGLA 1): Penaliza la omisión de cláusulas lógicas. Fuerza la correspondencia biyectiva entre conectores en lenguaje natural ("y" -> BinaryOp con "&&", "o" -> BinaryOp con "||"). Esto asegura que el modelo no descarte subcondiciones en expresiones booleanas compuestas.

Preservación de Gramática en Var (REGLA 3): Instruye explícitamente que los identificadores solo contienen "tag": "Var" y "name". Debido a la regla additionalProperties: false del esquema, previene que el LLM agregue atributos ilegales como "type" dentro de un nodo de variable.

Separación de Conceptos Funcionales (REGLA 5): Diferencia la abstracción (Lam: definición de función anónima con param, param_type y body) de la aplicación (App: llamada a una función con func y arg). Especifica que si la llamada es por nombre referencial, func debe ser un nodo Var.

Supresión de Marcadores de Formato (REGLA 6): Inhibe la emisión de bloques Markdown (json), permitiendo que la salida sea directamente interpretable por parsers JSON estándar.


*1.2 Estructura del Historial de Ejemplos (PROMPT_BASE_HISTORIAL)*

En lugar de incrustar ejemplos dentro de un bloque de texto plano, PROMPT_BASE_HISTORIAL estructura los casos de uso en pares conversacionales nativos (user / assistant):

Caso 1 (Condicional Compuesto): Muestra cómo descomponer un enunciado con conjunción lógica en un nodo raíz IfThenElse, cuyo campo cond es un BinaryOp ("&&") formado por comparaciones relacionales (">=" y "==").

Caso 2 (Aritmética y Flujo Alternativo): Enseña la construcción de árboles binarios aritméticos (BinaryOp con "+", "-", "*") anidados dentro de las ramas de bifurcación (then y else).

Caso 3 (Invocación Funcional mediante App y Var): Establece el patrón canónico para llamar funciones nombradas: el campo func referencia la variable de función (Var) y el campo arg contiene el valor de entrada (Literal).

Caso 4 (Abstracción Lambda Lam): Ilustra la sintaxis para definir funciones anónimas, enlazando el identificador del parámetro (param), su tipo atómico (param_type: {kind: "Base", name: "Int"}) y el cuerpo de evaluación (body).



**2) Especificación de Funciones**

*2.1 Función compilar(regla_negocio: str, max_tokens: int = 1024)*

Es la función central del pipeline encargada de transformar una cadena en lenguaje natural en un diccionario JSON validado:

Construcción del Contexto Dinámico: Clona la lista base PROMPT_BASE_HISTORIAL y añade en el turno final el rol user con la regla de negocio concreta a evaluar:


```
conversacion = list(PROMPT_BASE_HISTORIAL)
conversacion.append({"role": "user", "content": f"Regla de negocio:\n{regla_negocio}"})
```



Aplicación del Chat Template (tokenizer.apply_chat_template): Formatea los roles del historial con los tokens de control nativos del modelo (<|start_header_id|>, <|end_header_id|>, <|eot_id|>), asegurando que el modelo continúe la respuesta exactamente como el rol assistant.

Inferencia Determinista (model.generate): Configura la decodificación en modo voraz mediante temperature=0.0 y do_sample=False. Esto garantiza salidas reproducibles y estructuradas sin variabilidad estocástica.

Post-procesamiento y Sanitización: Extrae únicamente los nuevos tokens generados (inputs.input_ids.shape[1]:), decodifica el texto omitiendo tokens especiales (skip_special_tokens=True) y elimina posibles bloques residuales de markdown.

Validación Dual de Salida:

Sintáctica: Valida que el texto sea un objeto JSON bien formado mediante json.loads(ast_raw).

Semántica formal: Valida la estructura y tipado contra el esquema STLC mediante validate(instance=ast_json, schema=schema). Si el AST incumple alguna restricción formal, captura la excepción ValidationError y muestra la ruta exacta del nodo conflictivo.

*2.2 Validación de Tipos de Orden Superior (Arrow Type)*

Para reglas que involucran funciones como parámetros (orden superior), el sistema resuelve el campo recursivo param_type del esquema:

`Tipo flecha: { "kind": "Arrow", "from": { "kind": "Base", "name": "Int" }, "to": { "kind": "Base", "name": "Int" } }`

Esto asegura que el modelo comprende la distinción entre un tipo base (kind: "Base") y un tipo funcional flecha (kind: "Arrow", que exige las propiedades recursivas from y to).

In [3]:
import json
import torch
from jsonschema import validate, ValidationError

# 1. Definición del Prompt Único Universal
schema_str = json.dumps(schema, indent=2)

system_instruction = f"""Actúas como un compilador formal y determinista que traduce reglas de negocio en lenguaje natural a un Árbol de Sintaxis Abstracta (AST) en formato JSON, conforme al siguiente JSON Schema:
<JSON_SCHEMA>
{schema_str}
</JSON_SCHEMA>

REGLAS DE COMPILACIÓN OBLIGATORIAS:
1. EXHAUSTIVIDAD TOTAL: NUNCA omitas condiciones ni operaciones. Para conjunciones ("y", "and") usa "op": "&&". Para disyunciones ("o", "or") usa "op": "||".
2. TIPADO Y LITERALES:
   - `Literal`: siempre incluye "tag": "Literal", "value", y "type" con kind: "Base" ("Double", "Int", "Bool", "String").
3. IDENTIFICADORES:
   - `Var`: solo tiene "tag": "Var" y "name". NUNCA agregues la propiedad "type" a un nodo Var.
4. CONTROL DE FLUJO:
   - `IfThenElse`: contiene "tag": "IfThenElse", "cond", "then", "else".
5. PROGRAMACIÓN FUNCIONAL:
   - `Lam`: definición de función anónima con "tag": "Lam", "param", "param_type" (Type), y "body" (Expr).
   - `App`: invocación de función con "tag": "App", "func" (Expr), y "arg" (Expr). Si se invoca una función por su nombre, "func" DEBE ser un nodo "Var".
6. FORMATO DE SALIDA:
   - Devuelve ÚNICAMENTE el objeto JSON que representa la raíz del AST.
   - Sin explicaciones, comentarios ni bloques markdown (no usar ```json ni ```)."""

# Plantilla base con el catálogo canónico (Few-Shot unificado)
PROMPT_BASE_HISTORIAL = [
    {"role": "system", "content": system_instruction},

    # Caso 1: Condicional compuesto con operadores lógicos y comparaciones
    {
        "role": "user",
        "content": "Regla de negocio:\nSi la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', devolver true; caso contrario, false."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "tag": "IfThenElse",
            "cond": {
                "tag": "BinaryOp",
                "op": "&&",
                "left": {
                    "tag": "BinaryOp",
                    "op": ">=",
                    "left": {"tag": "Var", "name": "edad"},
                    "right": {"tag": "Literal", "type": {"kind": "Base", "name": "Int"}, "value": 18}
                },
                "right": {
                    "tag": "BinaryOp",
                    "op": "==",
                    "left": {"tag": "Var", "name": "estado"},
                    "right": {"tag": "Literal", "type": {"kind": "Base", "name": "String"}, "value": "ACTIVO"}
                }
            },
            "then": {"tag": "Literal", "type": {"kind": "Base", "name": "Bool"}, "value": True},
            "else": {"tag": "Literal", "type": {"kind": "Base", "name": "Bool"}, "value": False}
        }, indent=2)
    },

    # Caso 2: Operación aritmética compuesta en ramas condicionales
    {
        "role": "user",
        "content": "Regla de negocio:\nSi la antiguedad es menor a 2 o el riesgo es igual a 'ALTO', devolver 0.0; caso contrario, devolver antiguedad * 1.5."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "tag": "IfThenElse",
            "cond": {
                "tag": "BinaryOp",
                "op": "||",
                "left": {
                    "tag": "BinaryOp",
                    "op": "<",
                    "left": {"tag": "Var", "name": "antiguedad"},
                    "right": {"tag": "Literal", "type": {"kind": "Base", "name": "Double"}, "value": 2.0}
                },
                "right": {
                    "tag": "BinaryOp",
                    "op": "==",
                    "left": {"tag": "Var", "name": "riesgo"},
                    "right": {"tag": "Literal", "type": {"kind": "Base", "name": "String"}, "value": "ALTO"}
                }
            },
            "then": {"tag": "Literal", "type": {"kind": "Base", "name": "Double"}, "value": 0.0},
            "else": {
                "tag": "BinaryOp",
                "op": "*",
                "left": {"tag": "Var", "name": "antiguedad"},
                "right": {"tag": "Literal", "type": {"kind": "Base", "name": "Double"}, "value": 1.5}
            }
        }, indent=2)
    },

    # Caso 3: Aplicación de función nombrada (App canónico con Var)
    {
        "role": "user",
        "content": "Regla de negocio:\nAplicar la función calcularImpuesto al monto 100.0."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "tag": "App",
            "func": {
                "tag": "Var",
                "name": "calcularImpuesto"
            },
            "arg": {
                "tag": "Literal",
                "type": {"kind": "Base", "name": "Double"},
                "value": 100.0
            }
        }, indent=2)
    },

    # Caso 4: Definición de función lambda (Lam)
    {
        "role": "user",
        "content": "Regla de negocio:\nDefinir una función anónima que recibe x de tipo Int y devuelve x + 1."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "tag": "Lam",
            "param": "x",
            "param_type": {"kind": "Base", "name": "Int"},
            "body": {
                "tag": "BinaryOp",
                "op": "+",
                "left": {"tag": "Var", "name": "x"},
                "right": {"tag": "Literal", "type": {"kind": "Base", "name": "Int"}, "value": 1}
            }
        }, indent=2)
    }
]

# 2. Función Unificada de Compilación
def compilar(regla_negocio: str, max_tokens: int = 1024):
    conversacion = list(PROMPT_BASE_HISTORIAL)
    conversacion.append({
        "role": "user",
        "content": f"Regla de negocio:\n{regla_negocio}"
    })

    prompt = tokenizer.apply_chat_template(
        conversacion,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    ast_raw = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Limpieza de markdown residual
    if ast_raw.startswith("```json"):
        ast_raw = ast_raw[7:]
    if ast_raw.startswith("```"):
        ast_raw = ast_raw[3:]
    if ast_raw.endswith("```"):
        ast_raw = ast_raw[:-3]
    ast_raw = ast_raw.strip()

    print("=" * 60)
    print("AST GENERADO:")
    print("=" * 60)
    print(ast_raw)
    print("=" * 60)

    try:
        ast_json = json.loads(ast_raw)
        validate(instance=ast_json, schema=schema)
        print("✓ JSON sintáctico y Schema STLC 100% válidos.")
        return ast_json
    except json.JSONDecodeError as e:
        print(f"✗ Error sintáctico JSON: {e}")
    except ValidationError as e:
        print(f"✗ Error contra JSON Schema: {e.message}")
    return None

## **Llama 3.1 Instruct - 8B**

In [ ]:
# Instalar dependencias necesarias si aún no están en el entorno
!pip install -q -U transformers accelerate bitsandbytes

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_path = "/content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/llama-3.1-8b-instruct"

# Configuración de cuantización en 4-bit (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Cargar Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Cargar Modelo cuantizado en GPU
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

# Obtener límites de contexto / generación de tokens
model_max_pos = getattr(model.config, "max_position_embeddings", None)
tokenizer_max_len = getattr(tokenizer, "model_max_length", None)

print("-" * 50)
print(f"Dispositivo asignado al modelo: {model.device}")
print(f"Ventana de contexto máxima del modelo: {model_max_pos} tokens")
print(f"Límite configurado en el tokenizer: {tokenizer_max_len} tokens")
print("-" * 50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.8 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

--------------------------------------------------
Dispositivo asignado al modelo: cuda:0
Ventana de contexto máxima del modelo: 131072 tokens
Límite configurado en el tokenizer: 131072 tokens
--------------------------------------------------


En este banco de pruebas se evalúan los pilares teóricos y formales del Cálculo Lambda Simplemente Tipado ($\lambda^{\to}$ o STLC) extendido con constantes y tipos base, así como conceptos centrales del paradigma funcional:

**1) Transparencia Referencial y Expresiones como Ciudadanos de Primera Clase**

Reglas 1 y 2:

*Ausencia de sentencias imperativas (Statements vs. Expressions)*: No existen instrucciones de control de flujo clásicas (if/else procedurales que mutan estado o hacen return). El constructo IfThenElse se modela como una expresión ternaria pura: siempre evalúa a un valor y ambas ramas (then y else) deben tipar de forma homogénea ($T$).

*Composicionalidad y árboles de expresión:* En la Regla 2 se pone a prueba la evaluación composicional de operadores binarios (BinaryOp), donde las operaciones aritméticas complejas no se ejecutan secuencialmente, sino que se representan como árboles sintácticos libres de efectos secundarios (side-effect free).

**2) Abstracción Funcional y Ligadura de Variables (Lambda Abstraction & Variable Binding)**

Regla 3:

*Nociones de cálculo lambda atómico ($\lambda x : T . e$)*: Representado por el nodo Lam. Evalúa si el compilador entiende el concepto de abstracción pura: ligar un parámetro formal a un identificador (param) con un tipo explícito (param_type) para parametrizar un cómputo expresado en su cuerpo (body).

*Alcance Léxico (Lexical Scoping):* El identificador puntaje pasa de ser una variable libre a una variable ligada (bound variable) dentro del alcance del subárbol condicional.

**3) Aplicación de Funciones y Reducción (Function Application & Redex)**

Regla 4:

*Aplicación pura ($e_1 \; e_2$):* Representada por el constructor App. En el cálculo lambda, la aplicación es la única forma de consumir funciones y desencadenar reducción ($\beta$-reducción).

*Composición de funciones (Function Composition):* Al encadenar verificarAcceso (calcularNivel 42), se evalúa la asociatividad a izquierda de la aplicación funcional y la capacidad de tratar el resultado de una reducción como argumento directo de otra función: $(f \circ g)(x) \equiv f(g(x))$.

**4) Funciones de Primera Clase, Currificación y Clausuras (First-Class Functions, Currying & Closures)**

Regla 5:

*Currificación (Currying):* En cálculo lambda formal todas las funciones son unarias (aceptan exactamente un argumento). Para representar una función binaria (como multiplicar base por exponente), se currifica transformando $f(x, y)$ en una secuencia de abstracciones unarias anidadas: $\lambda \text{base} . (\lambda \text{exponente} . e)$.

*Funciones de Orden Superior (Higher-Order Functions):* La primera abstracción lambda no devuelve un valor primitivo, sino otra función como valor de retorno.

*Captura de entorno / Clausuras (Closures):* La lambda interna que recibe exponente debe retener en su entorno léxico la variable base, capturada de la abstracción exterior.

**5) Sistema de Tipos Simples (Simply Typed System)**

*Transversal a todas las reglas:*
1. Diferenciación entre tipos base y tipos funcionales:

    -Tipos atómicos/base ($\text{BaseType}$): Double, Int, Bool, String.
    
    -Tipos flecha ($\text{ArrowType}: T_1 \to T_2$): La capacidad de representar signaturas de orden superior donde el tipo mismo es recursivo (como se vio en la prueba con (Int -> Int)).

*Monoforfismo estricto:* Cada término en STLC tiene un tipo unívoco y determinable estáticamente, lo que garantiza formalmente la propiedad de Normalización Fuerte (todo programa bien tipado en STLC termina; no hay loops infinitos ni recursión no tipada sin combinadores de punto fijo explícitos).

In [ ]:
# Lista de las 5 reglas para estresar la gramática STLC
reglas_evaluacion = [
    # 1. Condicionales anidados con conectores mixtos (&& y ||)
    "Si la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', evaluar si el balance es mayor a 0.0 o tiene_garantia es true; si se cumple, devolver 1.0; caso contrario, devolver 0.0.",

    # 2. Aritmética multi-operador con precedencia y tipos Double
    "Si el salario es menor o igual a 50000.0, devolver (salario * 1.20) + 500.0; caso contrario, devolver salario - (salario * 0.05).",

    # 3. Abstracción Lambda con condicional interno (Lam + IfThenElse)
    "Definir una función anónima que recibe una variable puntaje de tipo Int y hace lo siguiente: si puntaje es mayor a 70, devuelve true; caso contrario, devuelve false.",

    # 4. Invocación compuesta anidada (App sobre App)
    "Aplicar la función verificarAcceso al resultado de aplicar la función calcularNivel al valor 42.",

    # 5. Función de orden superior currificada (Lam dentro de Lam)
    "Definir una función que recibe una base de tipo Double y retorna otra función que recibe un exponente de tipo Double y devuelve base * exponente."
]

resultados = []

print("INICIANDO EVALUACIÓN DE LAS 5 REGLAS...\n")

for i, regla in enumerate(reglas_evaluacion, 1):
    print(f"\n{'='*70}")
    print(f"▶ TEST {i}/5")
    print(f"Enunciado: {regla}")
    print(f"{'='*70}")

    ast = compilar(regla)

    if ast is not None:
        resultados.append((i, "EXITOSO", ast.get("tag", "Desconocido")))
    else:
        resultados.append((i, "FALLIDO", "Error en parseo o schema"))

# Resumen de ejecución
print("\n" + "#"*70)
print("RESUMEN DE EVALUACIÓN")
print("#"*70)
for test_num, estado, root_tag in resultados:
    print(f"Test {test_num}: {estado} | Nodo raíz: {root_tag}")
print("#"*70)

INICIANDO EVALUACIÓN DE LAS 5 REGLAS...


▶ TEST 1/5
Enunciado: Si la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', evaluar si el balance es mayor a 0.0 o tiene_garantia es true; si se cumple, devolver 1.0; caso contrario, devolver 0.0.
AST GENERADO:
{
  "tag": "IfThenElse",
  "cond": {
    "tag": "BinaryOp",
    "op": "&&",
    "left": {
      "tag": "BinaryOp",
      "op": ">=",
      "left": {
        "tag": "Var",
        "name": "edad"
      },
      "right": {
        "tag": "Literal",
        "type": {
          "kind": "Base",
          "name": "Int"
        },
        "value": 18
      }
    },
    "right": {
      "tag": "BinaryOp",
      "op": "==",
      "left": {
        "tag": "Var",
        "name": "estado"
      },
      "right": {
        "tag": "Literal",
        "type": {
          "kind": "Base",
          "name": "String"
        },
        "value": "ACTIVO"
      }
    }
  },
  "then": {
    "tag": "IfThenElse",
    "cond": {
      "tag": "BinaryOp",

Los resultados de estas primeras 5 inferencias son metodológica y técnicamente sobresalientes, especialmente considerando que estás utilizando un modelo de 8B cuantizado en 4-bit (llama-3.1-8b-instruct con BitsAndBytes NF4).

A continuación, los puntos clave que sustentan esta valoración desde la perspectiva de compiladores y modelado formal:

**1) Robustez frente a additionalProperties: false**
Uno de los fallos más comunes al forzar a un LLM a escupir JSON estructurado es el "derrame de atributos":

1. En los primeros intentos, el modelo tendía a meter "type" o metadatos inventados dentro de los nodos Var.

2. En esta corrida de 5 pruebas, el 100% de los nodos Var respetó la firma atómica {tag: "Var", name: "..."}. Que haya pasado la validación formal de jsonschema en todos los casos sin saltar errores de propiedades no permitidas demuestra que el in-context learning quedó perfectamente calibrado.

**2) Superación del sesgo de poda y conservación de la precedencia**

1. Lógica proposicional intacta: En el Test 1, el modelo no colapsó la lógica ni podó la segunda cláusula. Anidó correctamente la conjunción && y la disyunción ||, resolviendo además de forma idiomática el identificador booleano tiene_garantia como un operando directo de ||, sin agregar comparaciones redundantes (== true).

2. Árboles aritméticos coherentes: En el Test 2, entendió la jerarquía de operaciones al construir (salario * 1.20) + 500.0. Ubicó la suma en la raíz y la multiplicación en la rama izquierda (left), preservando el orden de evaluación sin aplanar la expresión.

**3) Ausencia de sesgo de anclaje en el nodo raíz**

Un riesgo crítico cuando se usan pocos ejemplos (few-shot) es que el modelo sobreajuste al constructor del primer ejemplo (en este caso, IfThenElse).

1. En el Test 3 no forzó un condicional en la raíz: detectó el intent declarativo ("Definir una función anónima...") y emitió un nodo Lam, relegando el IfThenElse al campo body.

2. En el Test 4 y Test 5 identificó correctamente que la raíz correspondía a App y a una secuencia currificada Lam -> Lam, respectivamente. Esto confirma que el modelo generaliza la gramática abstracta en lugar de memorizar una plantilla rígida.

**4) Resolución recursiva del sistema de tipos (Tipos Base vs. Arrow Type)**

El manejo de los tipos fue impecable:

1. Tipó con consistencia los literales numéricos (diferenciando Int de Double según la presencia de punto decimal o contexto).

2. En el caso de orden superior, no colapsó la signatura a un string plano como "Int -> Int". Construyó la estructura recursiva válida para el AST:

    ```
    "param_type": {
      "kind": "Arrow",
      "from": { "kind": "Base", "name": "Int" },
      "to": { "kind": "Base", "name": "Int" }
    }
    ```

Esto demuestra que el modelo internalizó la definición inductiva del tipo en el esquema.

**Conclusión técnica**

Para un modelo de 8 mil millones de parámetros, el prompt unificado logró convertirlo en un front-end de compilador determinista y confiable. Lograste desacoplar la variabilidad del lenguaje natural y mapearla 1:1 a la semántica formal de STLC sin requerir un fine-tuning dedicado, lo cual valida por completo la arquitectura de few-shot nativo por roles que implementaste.

Esta inferencia sirve para auditar la frontera más compleja de la gramática formal: la recursión en el sistema de tipos y las funciones de orden superior (Higher-Order Functions).

A diferencia de las cinco anteriores (que estresaban términos y expresiones dentro de Expr), esta prueba evalúa si el modelo comprende la definición inductiva de Type en el cálculo lambda simplemente tipado (STLC):

**1) El desafío para el compilador**

*Enunciado:*"Definir una función anónima que recibe un transformador de tipo (Int -> Int) y devuelve el resultado de aplicar transformador al valor 10."

*Lo que pone a prueba en el AST:*
1. Tipos de primer orden vs. tipos flecha: En las pruebas anteriores, los parámetros eran tipos base atómicos ({"kind": "Base", "name": "Int"}). Aquí, el parámetro formal transformador no es un dato escalar, sino una función ($T = \text{Int} \to \text{Int}$).

2. Estructura recursiva del JSON Schema: El modelo debe evitar la tentación de serializar el tipo como un string plano (e.g. "param_type": "Int -> Int", lo cual rompería el validador) y en su lugar construir el árbol inductivo:

    ```
    "param_type": {
      "kind": "Arrow",
      "from": { "kind": "Base", "name": "Int" },
      "to": { "kind": "Base", "name": "Int" }
    }
    ```




3. Consumo de funciones de primer orden: En el cuerpo (body), el modelo debe generar un nodo App donde el campo func sea la variable ligada transformador (Var) y el argumento arg sea el literal entero 10 (Literal).

**2) Conceptos de Programación Funcional evaluados**

*Funciones como Ciudadanos de Primera Clase (First-Class Functions):*
Demuestra que el sistema no limita las funciones al rol de meras rutinas a invocar, sino que pueden ser pasadas legítimamente como argumentos a otras abstracciones.

*Orden Superior Puro ($\lambda f : (T_1 \to T_2) . f \; e$)*: Es la signatura arquetípica de combinadores y operadores funcionales (análogo a un map unario o a la aplicación de un operador diferencial).

*Correspondencia de Curry-Howard a nivel tipos:* En la lógica intuicionista, el tipo flecha $A \to B$ modela la implicación lógica ($A \implies B$). Esta prueba confirma que el compilador puede instanciar premisas complejas en el contexto tipado del sistema.

**3) Propósito del bloque de inspección posterior**

El código posterior a la compilación añade una auditoría semántica fina:

1. No se conforma con que el JSON sea válido (validate(ast, schema)).
2. Inspecciona programáticamente el campo param_type para garantizar que el discriminador polimórfico kind haya conmutado a "Arrow" y haya instanciado correctamente las ramas from y to.

In [ ]:
# Regla diseñada para estresar param_type con kind: "Arrow"
regla_arrow_test = "Definir una función anónima que recibe un transformador de tipo (Int -> Int) y devuelve el resultado de aplicar transformador al valor 10."

print("Probando compilación de tipo funcional de orden superior (Arrow Type)...\n")
ast_arrow = compilar(regla_arrow_test)

# Inspección específica de la estructura del tipo generado
if ast_arrow and ast_arrow.get("tag") == "Lam":
    param_t = ast_arrow.get("param_type", {})
    print("\nInspección de tipo de parámetro:")
    print(f"- Kind detectado: {param_t.get('kind')}")
    if param_t.get("kind") == "Arrow":
        print(f"  ✓ from: {param_t.get('from')}")
        print(f"  ✓ to:   {param_t.get('to')}")
    else:
        print("  ✗ No generó kind 'Arrow'")

Probando compilación de tipo funcional de orden superior (Arrow Type)...

AST GENERADO:
{
  "tag": "Lam",
  "param": "transformador",
  "param_type": {
    "kind": "Arrow",
    "from": {
      "kind": "Base",
      "name": "Int"
    },
    "to": {
      "kind": "Base",
      "name": "Int"
    }
  },
  "body": {
    "tag": "App",
    "func": {
      "tag": "Var",
      "name": "transformador"
    },
    "arg": {
      "tag": "Literal",
      "type": {
        "kind": "Base",
        "name": "Int"
      },
      "value": 10
    }
  }
}
✓ JSON sintáctico y Schema STLC 100% válidos.

Inspección de tipo de parámetro:
- Kind detectado: Arrow
  ✓ from: {'kind': 'Base', 'name': 'Int'}
  ✓ to:   {'kind': 'Base', 'name': 'Int'}


Este resultado es la prueba de fuego definitiva del compilador y confirma que el sistema superó la barrera más compleja del esquema: el soporte recursivo de tipos de orden superior.   

El análisis técnico destaca varios logros esenciales:

**1) Manejo del tipo recursivo (Arrow Type)**

El mayor riesgo en esta prueba era que el modelo aplanara la signatura a un string como "Int -> Int" o que inventara claves fuera de especificación. En cambio:   
1. Reconoció que el parámetro no es un dato primitivo, sino una función.   
2. Construyó el subárbol inductivo exacto exigido por el esquema: kind: "Arrow", separando dominio (from) y codominio (to) como nodos kind: "Base" tipados a Int.   

**2) Semántica funcional del cuerpo (body)**

1. Ligadura correcta: Enlazó la variable ligada transformador (Var) directamente como el operador en el campo func del nodo App.   
2. Argumento canónico: Construyó el argumento de evaluación (arg) como un Literal primitivo de tipo Int con valor 10.   
3. Cumplimiento estricto: No generó propiedades no permitidas (additionalProperties: false) en ninguno de los nodos.   

**3) Generalización genuina**

En el historial de few-shot inicial no le habías dado un ejemplo idéntico de kind: "Arrow" como parámetro de una lambda. El modelo logró deducir la estructura válida combinando las reglas del system_prompt con la definición de definitions.Type del JSON Schema provisto.

# **Qwen 2.5 Instruct**

In [4]:
!pip install -U bitsandbytes transformers accelerate

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Liberar memoria VRAM del modelo anterior (Llama)
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer
gc.collect()
torch.cuda.empty_cache()

# 2. Ruta local en Drive de Qwen 2.5
qwen_model_path = "/content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/qwen-2.5-7b-instruct"

# 3. Configuración de cuantización en 4-bit (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# 4. Cargar Tokenizer de Qwen asegurando el pad_token_id
tokenizer = AutoTokenizer.from_pretrained(qwen_model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 5. Cargar Modelo en GPU
model = AutoModelForCausalLM.from_pretrained(
    qwen_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

print("-" * 50)
print(f"Modelo cargado: {qwen_model_path}")
print(f"Dispositivo asignado: {model.device}")
print(f"Ventana de contexto máxima: {getattr(model.config, 'max_position_embeddings', None)} tokens")
print("-" * 50)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

--------------------------------------------------
Modelo cargado: /content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/qwen-2.5-7b-instruct
Dispositivo asignado: cuda:0
Ventana de contexto máxima: 32768 tokens
--------------------------------------------------


In [5]:
# Lista de las 6 reglas canónicas de evaluación
reglas_evaluacion = [
    # 1. Condicionales anidados con conectores mixtos (&& y ||)
    "Si la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', evaluar si el balance es mayor a 0.0 o tiene_garantia es true; si se cumple, devolver 1.0; caso contrario, devolver 0.0.",

    # 2. Aritmética multi-operador con precedencia y tipos Double
    "Si el salario es menor o igual a 50000.0, devolver (salario * 1.20) + 500.0; caso contrario, devolver salario - (salario * 0.05).",

    # 3. Abstracción Lambda con condicional interno (Lam + IfThenElse)
    "Definir una función anónima que recibe una variable puntaje de tipo Int y hace lo siguiente: si puntaje es mayor a 70, devuelve true; caso contrario, devuelve false.",

    # 4. Invocación compuesta anidada (App sobre App)
    "Aplicar la función verificarAcceso al resultado de aplicar la función calcularNivel al valor 42.",

    # 5. Función de orden superior currificada (Lam dentro de Lam)
    "Definir una función que recibe una base de tipo Double y retorna otra función que recibe un exponente de tipo Double y devuelve base * exponente.",

    # 6. Orden superior con Arrow Type en param_type
    "Definir una función anónima que recibe un transformador de tipo (Int -> Int) y devuelve el resultado de aplicar transformador al valor 10."
]

resultados = []

print("INICIANDO EVALUACIÓN DE LAS 6 REGLAS EN QWEN 2.5...\n")

for i, regla in enumerate(reglas_evaluacion, 1):
    print(f"\n{'='*70}")
    print(f"▶ TEST {i}/6")
    print(f"Enunciado: {regla}")
    print(f"{'='*70}")

    ast = compilar(regla)

    if ast is not None:
        root_tag = ast.get("tag", "Desconocido")
        # Inspección extra para el test de orden superior (Test 6)
        if i == 6 and root_tag == "Lam":
            param_t = ast.get("param_type", {})
            kind = param_t.get("kind")
            print(f"\n[Auditoría Test 6] Kind detectado en param_type: {kind}")
            if kind == "Arrow":
                print(f"  ✓ from: {param_t.get('from')}")
                print(f"  ✓ to:   {param_t.get('to')}")
            else:
                print("  ✗ No generó kind 'Arrow'")

        resultados.append((i, "EXITOSO", root_tag))
    else:
        resultados.append((i, "FALLIDO", "Error en parseo o schema"))

# Resumen final de ejecución
print("\n" + "#"*70)
print("RESUMEN DE EVALUACIÓN (QWEN 2.5)")
print("#"*70)
for test_num, estado, root_tag in resultados:
    print(f"Test {test_num}: {estado} | Nodo raíz: {root_tag}")
print("#"*70)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INICIANDO EVALUACIÓN DE LAS 6 REGLAS EN QWEN 2.5...


▶ TEST 1/6
Enunciado: Si la edad es mayor o igual a 18 y el estado es igual a 'ACTIVO', evaluar si el balance es mayor a 0.0 o tiene_garantia es true; si se cumple, devolver 1.0; caso contrario, devolver 0.0.
AST GENERADO:
{
  "tag": "IfThenElse",
  "cond": {
    "tag": "BinaryOp",
    "op": "&&",
    "left": {
      "tag": "BinaryOp",
      "op": ">=",
      "left": {
        "tag": "Var",
        "name": "edad"
      },
      "right": {
        "tag": "Literal",
        "type": {
          "kind": "Base",
          "name": "Int"
        },
        "value": 18
      }
    },
    "right": {
      "tag": "BinaryOp",
      "op": "==",
      "left": {
        "tag": "Var",
        "name": "estado"
      },
      "right": {
        "tag": "Literal",
        "type": {
          "kind": "Base",
          "name": "String"
        },
        "value": "ACTIVO"
      }
    }
  },
  "then": {
    "tag": "IfThenElse",
    "cond": {
      "tag":